# Selection Tool Functions 

## imports

In [1]:
import pandas as pd
import json
import yaml
import jsonschema
from jsonschema import validate
from jsonschema import Draft7Validator, ValidationError
from datetime import date, datetime
from typing import Dict, List, Union

## Constants

In [2]:
SPORT = 'Bobsleigh'
# Folders
DATA_FOLDER_NAME = 'data'
RULE_FOLDER_NAME = 'rules'
SCHEMA_FOLDER_NAME = 'schemas'
# Files
DATA_FILE_NAME = 'data_2026-01-07.csv'
RULE_FILE_NAME = 'Ski_jumping_qualification_rules.yaml'
DATA_SCHEMA_FILE_NAME = 'dataschema.json'
RULE_SCHEMA_FILE_NAME = 'ruleschema.json'
# Paths
DATA_PATH = f'../{DATA_FOLDER_NAME}/{DATA_FILE_NAME}'
RULE_PATH = f'../{RULE_FOLDER_NAME}/{RULE_FILE_NAME}'
DATA_SCHEMA_PATH = f'../{SCHEMA_FOLDER_NAME}/{DATA_SCHEMA_FILE_NAME}'
RULE_SCHEMA_PATH = f'../{SCHEMA_FOLDER_NAME}/{RULE_SCHEMA_FILE_NAME}'

In [3]:
print(DATA_PATH)

../data/data_2026-01-07.csv


## read functions 

In [4]:
# Function to read data from CSV
def load_data(data_path):
    data = pd.read_csv(data_path)
    data = data.where(pd.notnull(data), None)
    return data

# Function to read JSON schema
def load_json_schema(schema_path):
    with open(schema_path, 'r') as f:
        return json.load(f)

# Function to read YAML rules
def load_yaml_rules(rules_path):
    with open(rules_path, 'r') as f:
        return yaml.safe_load(f)


In [5]:
# Load all data and schemas
data = load_data(DATA_PATH)
data_schema = load_json_schema(DATA_SCHEMA_PATH)
rule_schema = load_json_schema(RULE_SCHEMA_PATH)
rules = load_yaml_rules(RULE_PATH)

## Validation Functions

### data validation

In [6]:
# A function that validates data against a schema
def validate_data(data, schema):
    data = data.to_dict(orient='records')
    try:
        validate(instance=data, schema=schema)
        print("Data is valid.")
        return True
    except jsonschema.exceptions.ValidationError as err:
        print("Data is invalid:", err)
        return False

# A function that takes data and the schema and removes all invalid columns in the data frame based on the schema
def clean_data(data, schema):
    valid_columns = [prop for prop in schema['properties'].keys() if prop in data.columns]
    # remove all rows with Team Members == Yes
    cleaned_data = data[valid_columns]
    cleaned_data = cleaned_data[cleaned_data["Team Members"] != "Yes"]
    # remove coumn "Team Members"
    cleaned_data = cleaned_data.drop(columns=["Team Members"])
    return cleaned_data

In [7]:
# Extract the item schema since data_schema is for an array
item_schema = data_schema.get('items', data_schema)
cleaned_data = clean_data(data, item_schema)
vality = validate_data(cleaned_data, data_schema)

Data is valid.


### rule validation

In [8]:
def normalize_rules(obj):
    """
    Recursively normalize a dict representing Alpine Skiing Olympic rules:
    1. Convert empty strings or None-like values to Python None.
    2. Convert 'created at' and 'updated at' dates to ISO strings.
    3. Convert 'version' to string.
    """
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            # Step 1: Normalize recursively
            v = normalize_rules(v)

            # Step 2: Handle dates
            if k in ("created at", "updated at") and isinstance(v, (date, datetime)):
                new_obj[k] = v.isoformat()
            # Step 3: Handle version field
            elif k == "version" and isinstance(v, (float, int)):
                new_obj[k] = str(v)
            # Step 4: Convert empty string or None to Python None
            elif v == "" or v is None:
                new_obj[k] = None
            else:
                new_obj[k] = v
        return new_obj
    elif isinstance(obj, list):
        return [normalize_rules(v) for v in obj]
    elif obj == "" or obj is None:
        return None
    else:
        return obj

    
def validate_rules(data: dict, schema: dict) -> bool:
    """
    Validate a Python dict against a JSON Schema.
    Prints all validation errors if they exist.
    Returns True if validation passed, False otherwise.
    """
    validator = Draft7Validator(schema)
    errors = sorted(validator.iter_errors(data), key=lambda e: e.path)

    if not errors:
        print("✅ Validation passed!")
        return True

    print("❌ Validation failed with the following errors:\n")
    for err in errors:
        path = ".".join([str(p) for p in err.path])
        print(f"- Path: {path or 'root'}")
        print(f"  Message: {err.message}\n")
    return False


In [9]:
rules = normalize_rules(rules)
is_valid = validate_rules(rules, rule_schema)

❌ Validation failed with the following errors:

- Path: rule_tree.main criterias.0.conditions.0.date
  Message: 2025 is not of type 'array'

- Path: rule_tree.main criterias.4.conditions.0.date
  Message: 2026 is not of type 'array'

- Path: rule_tree.main criterias.5.conditions.0.date
  Message: 2026 is not of type 'array'

- Path: rule_tree.main criterias.6.conditions.0.date
  Message: 2025 is not of type 'array'



## Apply Rules

In [10]:
def filter_by_date(
    df: pd.DataFrame,
    date_rule: Union[int, List[str]]
) -> pd.DataFrame:
    """
    Filters dataframe by either a single year or a date range.
    """
    if isinstance(date_rule, int):
        return df[df["Year"] == date_rule]

    start_date, end_date = pd.to_datetime(date_rule)
    dates = pd.to_datetime(df["Date"])
    return df[(dates >= start_date) & (dates <= end_date)]

def condition_is_met(
    athlete_results: pd.DataFrame,
    condition: Dict
) -> bool:
    """
    Returns True if the athlete meets a single condition.
    """
    filtered = athlete_results.copy()

    # Ensure Rank is numeric
    filtered["Rank"] = pd.to_numeric(filtered["Rank"], errors="coerce")

    # Competition filter
    filtered = filtered[
        filtered["Comp.SetDetail"] == condition["competition"]
    ]

    # Date filter
    filtered = filter_by_date(filtered, condition["date"])

    # Performance (rank) filter
    min_rank, max_rank = condition["performance"]
    filtered = filtered[
        filtered["Rank"].between(min_rank, max_rank)
    ]

    # Count requirement
    return len(filtered) >= condition["count_at_least"]



def main_criteria_is_met(
    athlete_results: pd.DataFrame,
    main_criteria: Dict
) -> bool:
    """
    Returns True if ALL conditions of a main criteria are met.
    """
    return all(
        condition_is_met(athlete_results, condition)
        for condition in main_criteria["conditions"]
    )

def evaluate_athlete_qualification(
    df: pd.DataFrame,
    rule_tree: Dict
) -> pd.DataFrame:
    """
    Returns a dataframe:
    index   -> athlete
    columns -> main criteria descriptions
    values  -> True / False
    """
    results = {}

    for athlete, athlete_df in df.groupby("Person"):
        athlete_results = {}

        for criteria in rule_tree["main criterias"]:
            criteria_name = criteria["description"]

            athlete_results[criteria_name] = main_criteria_is_met(
                athlete_df,
                criteria
            )

        results[athlete] = athlete_results

    return pd.DataFrame.from_dict(results, orient="index")


In [11]:
# filter cleaned data by SPORT
filtered_data = cleaned_data[cleaned_data['Sport'] == SPORT]
qualification_matrix = evaluate_athlete_qualification(
    df=filtered_data,
    rule_tree=rules["rule_tree"]
)

KeyError: 'Person'

In [ ]:
# sort by the number of criteria met descending
qualification_matrix['Total_Met'] = qualification_matrix.sum(axis=1)
qualification_matrix = qualification_matrix.sort_values(by='Total_Met', ascending=False)
# save qualification matrix to csv
qualification_matrix.to_csv('qualification_matrix.csv')